In [254]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [255]:
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as stop_words,TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [256]:
train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


# Question 1: Sum of the occurrences of the most frequent option and the least frequent option

In [257]:
freq_dist=train.groupby('answer')['answer'].count()
freq_dist

answer
A    369
B    490
C    459
D    358
E    324
Name: answer, dtype: int64

In [258]:
print(f"Sum of the occurrences of the most frequent option and the least frequent option: {freq_dist.max()+freq_dist.min()}")

Sum of the occurrences of the most frequent option and the least frequent option: 814


# Question 2: Total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv

In [259]:
train['clean_prompt']=train['prompt'].str.lower().apply(lambda x: str(x).translate(str.maketrans("", "", string.punctuation)))
res=train['clean_prompt'].str.cat(sep=' ')
print(f"Total number of unique words: {len(set(res.split()))}")

Total number of unique words: 859


# Question 3: English Stop Words

In [260]:
vec=TfidfVectorizer(stop_words='english')
text=train.loc[0,'clean_prompt']
filtered=[word for word in text.split() if word.lower() not in ENGLISH_STOP_WORDS]
no_stop_word=vec.fit_transform([text])
print(f"Total Filtered Words for Row 1: {len(filtered)}")

Total Filtered Words for Row 1: 13


# Question 4: Total number of feature columns (vocabulary size) generated by the TfidfVectorizer

In [261]:
vec2=TfidfVectorizer(stop_words='english')
l=(
    train['prompt'] + ' ' +
    train['A'] + ' ' +
    train['B'] + ' ' +
    train['C'] + ' ' +
    train['D'] + ' ' +
    train['E']
)
voc=vec2.fit_transform(l)
print(f"Total number of feature columns: {voc.shape[1]}")

Total number of feature columns: 2762


# Question 5: Cosine Similarity between Prompt and Option A for Row 1

In [262]:
mat1=vec2.transform([train.loc[0,'prompt']])
mat2=vec2.transform([train.loc[0,'A']])
print(f"Cosine Similarity between Prompt and Option A: {cosine_similarity(mat1, mat2)[0,0]:.4f}")

Cosine Similarity between Prompt and Option A: 0.2720


# Question 6: Percentage of instances where the option with the highest cosine similarity matches the correct answer.   

In [330]:
m1=vec2.transform(train['prompt'])
sims=[]
for i in ['A','B','C','D','E']:
    m2=vec2.transform(train[i])
    sim=cosine_similarity(m1, m2)
    sims.append(np.diag(sim))

sims=np.column_stack(sims)

predicted=np.array(['A','B','C','D','E'])[np.argmax(sims,axis=1)]

accuracy=(predicted == train['answer']).mean() * 100

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 13.55%


# Question 9: Overall MAP@3 score of "Majority Class" baseline on train.csv

In [264]:
freq_dist.sort_values(inplace=True,ascending=False)
first=freq_dist.index[0]
second=freq_dist.index[1]
third=freq_dist.index[2]
temp_train=train.copy()
temp_train.head()

,id,prompt,A,B,C,D,E,answer,clean_prompt
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer what is martin h...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is acceleratorbased lightion fusion
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,determine the correct option what is the term ...
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,select the most accurate option what is martin...
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,identify the correct statement what is the con...


In [265]:
pred=first+' '+second+' '+third
temp_train['answer']=pred
temp_train.head()

,id,prompt,A,B,C,D,E,answer,clean_prompt
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B C A,pick the best possible answer what is martin h...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,B C A,what is acceleratorbased lightion fusion
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,B C A,determine the correct option what is the term ...
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B C A,select the most accurate option what is martin...
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,B C A,identify the correct statement what is the con...


In [303]:
def map_at_3(true,pred):
    scores=[]
    for actual,preds in zip(true,pred):
        score=0.0
        for rank,pred in enumerate(preds.split(' '),start=1):
            if pred==actual:
                score=1.0/rank
                break
        scores.append(score)
    return np.mean(scores)

In [304]:
map3=map_at_3(train['answer'],temp_train['answer'])
print(map3)

0.42125


# Question 10: Final Average MAP@3 score of Tf-Idf pipeline

In [333]:
options=np.array(['A','B','C','D','E'])

top3_index=np.argsort(-sims,axis=1)[:,:3]
top3_preds=options[top3_index]
top3_preds=[' '.join(row) for row in top3_preds]

map3=map_at_3(train['answer'],top3_preds)
print(f"MAP@3 Score: {map3:.6f}")

MAP@3 Score: 0.291500
